# Simulation-Based Power Analysis Side-Channel Attack on AES
## Interactive Jupyter Notebook

Run cells top-to-bottom. Each cell is independent and fully explained.

**Sections:**
1. Setup & imports
2. S-Box and Hamming Weight model
3. Power trace simulation (all 16 bytes)
4. CPA attack on one byte
5. Success rate vs trace count
6. Full 16-byte key recovery
7. Countermeasures

In [ ]:
import sys, os
sys.path.insert(0, '../src')

import numpy as np
import matplotlib.pyplot as plt

from aes_sim import (
    SBOX, hamming_weight, subbytes_output,
    simulate_power_traces, correlation_power_analysis,
    leak_time_for_byte, BYTE_LEAK_SPACING,
)

plt.style.use('dark_background')
plt.rcParams['figure.facecolor'] = '#0D1117'
plt.rcParams['axes.facecolor']   = '#161B22'

BLUE, RED, GREEN = '#58A6FF', '#F78166', '#3FB950'

# Simulation constants
BASE_LEAK  = 5     # byte 0 leaks at t=5, byte b at t=5+b*3
TIME_PTS   = 70    # must be > 5 + 15*3 = 50  (we use 70 for margin)

print('Ready. TIME_PTS =', TIME_PTS)
print('Byte-0 leak at t =', leak_time_for_byte(0, BASE_LEAK))
print('Byte-15 leak at t =', leak_time_for_byte(15, BASE_LEAK))

## 1. AES S-Box & Hamming Weight

In [ ]:
# HW distribution of all 256 SubBytes outputs for an example key byte
example_key_byte = 0xE8
hw_vals = [hamming_weight(subbytes_output(p, example_key_byte)) for p in range(256)]
counts  = np.bincount(hw_vals, minlength=9)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
bars = ax.bar(range(9), counts, color=BLUE, alpha=0.85)
ax.bar_label(bars, fmt='%d', color='white', fontsize=9)
ax.set_title(f'HW of SubBytes(p XOR 0x{example_key_byte:02X})', color='white')
ax.set_xlabel('Hamming Weight'); ax.set_ylabel('Count'); ax.set_xticks(range(9))

ax2 = axes[1]
sbox_hw = np.vectorize(hamming_weight)(np.array(SBOX).reshape(16, 16))
im = ax2.imshow(sbox_hw, cmap='plasma', vmin=0, vmax=8)
plt.colorbar(im, ax=ax2, label='HW of S-Box output')
ax2.set_title('S-Box HW Heatmap (all 256 inputs)', color='white')

plt.tight_layout(); plt.show()
print(f'Mean HW = {np.mean(hw_vals):.2f},  Std = {np.std(hw_vals):.2f}')

## 2. Generate Power Traces (all 16 bytes leak)

In [ ]:
# ── Parameters you can change ─────────────────────────────────────────
N_TRACES  = 300
NOISE_STD = 0.8

RNG = np.random.default_rng(42)
TRUE_KEY   = RNG.integers(0, 256, 16, dtype=np.uint8)
PLAINTEXTS = RNG.integers(0, 256, (N_TRACES, 16), dtype=np.uint8)

print(f'Secret key: {" ".join(f"0x{b:02X}" for b in TRUE_KEY)}')

# simulate_power_traces embeds leakage for ALL 16 bytes
traces = simulate_power_traces(
    PLAINTEXTS, TRUE_KEY,
    noise_std=NOISE_STD,
    time_points=TIME_PTS,
    leak_time=BASE_LEAK,
)
print(f'Traces shape: {traces.shape}  (N traces × T time samples)')

fig, ax = plt.subplots(figsize=(13, 4))
for i in range(min(25, N_TRACES)):
    ax.plot(traces[i], color=BLUE, alpha=0.15, linewidth=0.7)
ax.plot(traces.mean(axis=0), color=GREEN, linewidth=2.5, label='Mean trace')

# Mark all 16 leak points
for b in range(16):
    lt = leak_time_for_byte(b, BASE_LEAK)
    ax.axvline(lt, color=RED, linewidth=0.8, alpha=0.5)
ax.axvline(leak_time_for_byte(0, BASE_LEAK), color=RED, linewidth=2,
           linestyle='--', label='Byte leak points')

ax.set_title(f'Power Traces (N={N_TRACES}, σ={NOISE_STD})', color='white')
ax.set_xlabel('Time Sample'); ax.set_ylabel('Power (a.u.)')
ax.legend(); ax.grid(True); plt.show()

## 3. CPA Attack — Single Byte

In [ ]:
BYTE_IDX = 0   # change to 1..15 to attack a different byte

corr_matrix, best_guess = correlation_power_analysis(
    PLAINTEXTS, traces, byte_idx=BYTE_IDX)
recovered = int(np.argmax(best_guess))

print(f'True key byte {BYTE_IDX}:  0x{TRUE_KEY[BYTE_IDX]:02X}')
print(f'Recovered byte {BYTE_IDX}: 0x{recovered:02X}')
print(f'Attack: {"SUCCESS" if recovered == TRUE_KEY[BYTE_IDX] else "FAIL"}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5),
                          gridspec_kw={'width_ratios': [3, 1]})
ax = axes[0]
im = ax.imshow(np.abs(corr_matrix), aspect='auto', cmap='inferno',
               origin='lower', vmin=0, vmax=0.6)
ax.axhline(TRUE_KEY[BYTE_IDX], color=GREEN, lw=2, ls='--',
           label=f'True key 0x{TRUE_KEY[BYTE_IDX]:02X}')
plt.colorbar(im, ax=ax, label='|Correlation|')
ax.set_title('CPA Correlation Heatmap', color='white')
ax.set_xlabel('Time Sample'); ax.set_ylabel('Key Hypothesis (0-255)')
ax.legend()

ax2 = axes[1]
colors = [RED]*256
colors[TRUE_KEY[BYTE_IDX]] = GREEN
ax2.barh(range(256), best_guess, color=colors, height=1)
ax2.axhline(TRUE_KEY[BYTE_IDX], color=GREEN, lw=2, ls='--')
ax2.set_title('Max |r| per hypothesis', color='white')
ax2.set_xlabel('Max |Correlation|'); ax2.set_ylim(-1, 256)

plt.tight_layout(); plt.show()

## 4. Success Rate vs Trace Count

In [ ]:
trace_counts = [5, 10, 20, 30, 50, 75, 100, 150, 200]
RUNS = 10

rng2 = np.random.default_rng(7)
big_pts = rng2.integers(0, 256, (500, 16), dtype=np.uint8)
big_key = rng2.integers(0, 256, 16, dtype=np.uint8)
big_tr  = simulate_power_traces(big_pts, big_key, noise_std=1.2,
                                 time_points=TIME_PTS, leak_time=BASE_LEAK)

rates = []
for n in trace_counts:
    ok = 0
    for _ in range(RUNS):
        idx = rng2.choice(500, n, replace=False)
        _, bg = correlation_power_analysis(big_pts[idx], big_tr[idx], 0)
        if np.argmax(bg) == big_key[0]: ok += 1
    rates.append(ok / RUNS * 100)
    print(f'N={n:4d}  {ok}/{RUNS}  ({rates[-1]:.0f}%)')

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(trace_counts, rates, 'o-', color=BLUE, lw=2, markersize=7,
        markerfacecolor=GREEN)
ax.fill_between(trace_counts, rates, alpha=0.1, color=BLUE)
ax.axhline(100, color=GREEN, lw=1, ls='--', alpha=0.5)
ax.set_title('CPA Success Rate vs Trace Count  (σ=1.2)', color='white')
ax.set_xlabel('N traces'); ax.set_ylabel('Success rate (%)')
ax.set_ylim(-5, 110); ax.grid(True); plt.show()

## 5. Full 16-Byte Key Recovery

In [ ]:
rng3 = np.random.default_rng(999)
pts3 = rng3.integers(0, 256, (500, 16), dtype=np.uint8)
key3 = rng3.integers(0, 256, 16, dtype=np.uint8)

# All 16 bytes leak — each at its own time slot
tr3  = simulate_power_traces(pts3, key3, noise_std=0.9,
                              time_points=TIME_PTS, leak_time=BASE_LEAK)

recovered3 = []
for b in range(16):
    _, bg = correlation_power_analysis(pts3, tr3, byte_idx=b)
    recovered3.append(int(np.argmax(bg)))
recovered3 = np.array(recovered3)
match3 = recovered3 == key3.astype(int)

print(f'True key  : {" ".join(f"0x{b:02X}" for b in key3)}')
print(f'Recovered : {" ".join(f"0x{b:02X}" for b in recovered3)}')
print(f'Match     : {" ".join("OK" if m else "XX" for m in match3)}')
print(f'Score     : {match3.sum()}/16 bytes correct')

fig, ax = plt.subplots(figsize=(14, 4))
x = np.arange(16); w = 0.35
ax.bar(x-w/2, key3, w, label='True Key', color=GREEN, alpha=0.9)
bar_clrs = [BLUE if m else RED for m in match3]
ax.bar(x+w/2, recovered3, w, label='Recovered', color=bar_clrs, alpha=0.9)
ax.set_xticks(x); ax.set_xticklabels([f'B{i}' for i in range(16)])
ax.set_title(f'Full AES-128 Key Recovery  ({match3.sum()}/16 correct)',
             color='white')
ax.set_xlabel('Key Byte Index'); ax.set_ylabel('Byte Value (0-255)')
ax.legend(); ax.grid(True, axis='y'); plt.show()